In [ ]:
import anomalib
print(anomalib.__version__)

In [ ]:

from pathlib import Path


from anomalib.data.utils import read_image
from anomalib.deploy import OpenVINOInferencer

In [ ]:
from typing import Any

import numpy as np

from PIL import Image
from torchvision.transforms import ToPILImage

from anomalib import TaskType
from anomalib.data import MVTec

from anomalib.deploy import ExportType, OpenVINOInferencer
from anomalib.engine import Engine


In [ ]:
from anomalib.data import Folder
from anomalib import TaskType

datamodule = Folder(
    name="GAN",
    root="/path/to/private-data/anomaly train Left outlet",
    normal_dir="normal",
    abnormal_dir="abnormal",
    normal_split_ratio=0.2,
    image_size=(400, 320),
    train_batch_size=32,
    eval_batch_size=32,
    task=TaskType.CLASSIFICATION,
)
datamodule.setup()

i, data = next(enumerate(datamodule.val_dataloader()))
print(data.keys())




In [ ]:
# Check image size
print(data["image"].shape)

In [ ]:
from anomalib.models import Padim
from anomalib.models import Ganomaly

model = Ganomaly(batch_size=32, n_features=64, latent_vec_size=100, extra_layers=0, add_final_conv_layer=True, wadv=1, wcon=50, wenc=1, lr=0.0002, beta1=0.5, beta2=0.999)

#model = Padim(    backbone="resnet18",    layers=["layer1", "layer2", "layer3"],)

In [ ]:
from anomalib.engine import Engine
from anomalib.utils.normalization import NormalizationMethod

engine = Engine(
    normalization=NormalizationMethod.MIN_MAX,
    threshold="F1AdaptiveThreshold",
    task=TaskType.CLASSIFICATION,
    image_metrics=["AUROC"],
    accelerator="auto",
    check_val_every_n_epoch=1,
    devices=1,
    max_epochs=100,
    num_sanity_val_steps=0,
    val_check_interval=1.0,
)

engine.fit(model=model, datamodule=datamodule)





In [ ]:
# Validation
test_results = engine.test(model=model, datamodule=datamodule)


In [ ]:


from anomalib.data.utils import read_image


##image_path = "/path/to/private-data/Croped_Pictures/train/Right/Inlet (3 marking)/example.bmp" # inlet 3
#image_path = "/path/to/private-data/Croped_Pictures/train/Right/Inlet (5 marking)/example.bmp"    # inlet 5
#image_path = "/path/to/private-data/Croped_Pictures/train/Right/Outlet/example.bmp"   # Outlet

#image = read_image(path=image_path)
#plt.imshow(image)
#image.shape

In [ ]:
from anomalib.deploy import ExportType

# Exporting model to OpenVINO
openvino_model_path = engine.export(
    model=model,
    export_type=ExportType.OPENVINO,
    
)

In [ ]:

from anomalib.data.utils import read_image
from anomalib.utils.visualization.image import ImageVisualizer, VisualizationMode
from PIL import Image
output_path = Path(engine.trainer.default_root_dir)
print(output_path)
metadata_path = openvino_model_path.parent / "metadata.json"
print(openvino_model_path.exists(), metadata_path.exists())
inferencer = OpenVINOInferencer(
    path=openvino_model_path,  # Path to the OpenVINO IR model.
    metadata=metadata_path,  # Path to the metadata file.
    device="CPU",  # We would like to run it on an Intel CPU.
)

In [ ]:
import glob, os
 
in3_score = []
in5_score = []
outlet_score = []

in3_lable = []
in5_lable = []
outlet_lable = []



path = "/path/to/private-data/Croped_Pictures/train/Left/*"
for directory_path in glob.glob(path):
    label=directory_path.split("\\")[-1]
    for image_path in glob.glob(os.path.join(directory_path,"*.bmp")):
        image = read_image(image_path)
        predictions = inferencer.predict(image=image)
        score = predictions.pred_score
        predicted_label = predictions.pred_label
        
        if label == "Inlet (3 marking)":
            in3_score.append(score)
            in3_lable.append(predicted_label)

            
        elif label == "Inlet (5 marking)":
            in5_score.append(score)
            in5_lable.append(predicted_label)
            
        elif label == "Outlet":
            outlet_score.append(score)
            outlet_lable.append(predicted_label)

print('Done')

In [ ]:
import glob, os
 
val_in3_score = []
val_in5_score = []
val_outlet_score = []
val_others_score = []

val_in3_lable = []
val_in5_lable = []
val_outlet_lable = []
val_others_lable = []



path = "/path/to/private-data/Croped_Pictures/validation/Left/*"
for directory_path in glob.glob(path):
    label=directory_path.split("\\")[-1]
    for image_path in glob.glob(os.path.join(directory_path,"*.bmp")):
        image = read_image(image_path)
        predictions = inferencer.predict(image=image)
        score = predictions.pred_score
        predicted_label = predictions.pred_label
        
        if label == "Inlet (3 marking)":
            val_in3_score.append(score)
            val_in3_lable.append(predicted_label)

            
        elif label == "Inlet (5 marking)":
            val_in5_score.append(score)
            val_in5_lable.append(predicted_label)
            
        elif label == "Outlet":
            val_outlet_score.append(score)
            val_outlet_lable.append(predicted_label)
            
        elif label == "Others":
            val_others_score.append(score)
            val_others_lable.append(predicted_label)

print('Done')

In [ ]:
in3_lable = np.array(in3_lable).astype('int')
#true_in3_lable = np.ones(in3_lable.shape)

in5_lable = np.array(in5_lable).astype('int')
#true_in5_lable = np.ones(in5_lable.shape)

outlet_lable = np.array(outlet_lable).astype('int')
#true_outlet_lable = np.zeros(outlet_lable.shape)

val_in3_lable = np.array(val_in3_lable).astype('int')
#val_true_in3_lable = np.ones(val_in3_lable.shape)

val_in5_lable = np.array(val_in5_lable).astype('int')
#val_true_in5_lable = np.ones(val_in5_lable.shape)

val_outlet_lable = np.array(val_outlet_lable).astype('int')
#val_true_outlet_lable = np.zeros(val_outlet_lable.shape)

val_others_lable = np.array(val_others_lable).astype('int')
#val_true_others_lable = np.ones(val_others_lable.shape)



In [ ]:
%matplotlib inline
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
plt.figure(figsize=(8, 6))

# Create the box plot
sns.boxplot(data=[outlet_score,val_outlet_score, val_others_score,in3_score, in5_score   ])

# Setting labels
plt.xlabel('Class')

plt.title('Predicted Scores for Different Classes')
plt.xticks([0, 1, 2, 3, 4], ['Outlet', 'Outlet (test)','Unmarked','Inlet type-1', 'Inlet Type-2' ])  # Label x-ticks with class names

# Show the plot
plt.show()


In [ ]:
val_others_score

In [ ]:

# Counting the occurrences of normal (0) and anomaly (1) in each class
normal_counts = np.array([np.count_nonzero(pred == 0) for pred in [in3_lable,  in5_lable, outlet_lable, val_outlet_lable, val_others_lable]])
anomaly_counts = np.array([np.count_nonzero(pred == 1) for pred in [in3_lable, in5_lable,  outlet_lable,val_outlet_lable, val_others_lable]])
confusion_matrix = np.vstack([normal_counts, anomaly_counts])
import pandas as pd

# Creating a DataFrame
class_labels = ['Inlet type -1',  'Inlet type-2', 'Outlet', 'Outlet (Test)', 'Unmarked']
df_cm = pd.DataFrame(confusion_matrix, index=['Normal Predicted', 'Anomaly Predicted'], columns=class_labels)


# Plotting
plt.figure(figsize=(10, 5))
sns.heatmap(df_cm, annot=True, fmt="d", cmap='Blues', cbar=False)
plt.title('Prediction Outcomes by Class')
#plt.xlabel('Class')
plt.ylabel('Prediction Type')
plt.show()